# Destabilizing Mutation → Protein Abundance Analysis\n\nTests whether destabilizing missense mutations reduce protein abundance using shared peptides from psm.tsv.\n\n**Logic:**  \nFor each destabilizing mutation (high AlphaMissense OR SPURS ddG) in protein X within a plex:\n1. Identify which TMT channels carry the mutation (GDC UUID → missense table)\n2. Find **shared peptides**: reference PSMs mapping to protein X (not -mut, not -comp)\n3. Compare their TMT intensities: mutant-carrier channels vs non-carrier channels in same plex\n4. `delta = log2(mut_channel / mean(WT_channels))` — negative = less abundant\n\nA **neutral control** (benign by AM + SPURS) runs identically as negative control."

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP      = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META     = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
MISSENSE     = "/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv"
RESULTS_BASE = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
PLEX_LIST    = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
OUT_DIR      = "/scratch/leduc.an/AAS_Evo/ANALYSIS/destabilization"

# Destabilizing thresholds
AM_THRESHOLD    = 0.564   # AlphaMissense likely_pathogenic
SPURS_THRESHOLD = 0.5     # SPURS ddG destabilizing
VAF_THRESHOLD   = 0.3
GNOMAD_MAX      = 0.01    # exclude common variants from destabilizing set

# Neutral control: common low-pathogenicity variants
# High gnomAD AF (>0.1) = common population variant, unlikely driver
# Low AlphaMissense (<0.1) = predicted benign
AM_BENIGN_MAX   = 0.1
GNOMAD_NEUTRAL_MIN = 0.1  # must be common in population

N_PLEXES = None   # None = all; set int to test on subset e.g. 5

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
    "tmt_126c":"126C","tmt_134n":"134N",
}

os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded")

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
if N_PLEXES:
    plex_ids = plex_ids[:N_PLEXES]

print(f"Plexes: {len(plex_ids)} | TMT map: {len(tmt):,} rows | GDC meta: {len(gdc):,} rows")

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print('Loading missense table...')
missense = pd.read_csv(MISSENSE, sep='\t', low_memory=False)
missense['am_pathogenicity'] = pd.to_numeric(missense['am_pathogenicity'], errors='coerce')
missense['spurs_ddg']        = pd.to_numeric(missense['spurs_ddg'], errors='coerce')
missense['VAF']              = pd.to_numeric(missense['VAF'], errors='coerce')
missense['gnomADe_AF']       = pd.to_numeric(missense['gnomADe_AF'], errors='coerce').fillna(0)
print(f'Total rows: {len(missense):,}')

base_ok   = (missense['VAF'] >= VAF_THRESHOLD) & (missense['gnomADe_AF'] < GNOMAD_MAX)
am_pos    = missense['am_pathogenicity'] >= AM_THRESHOLD    # NaN -> False
spurs_pos = missense['spurs_ddg']        >= SPURS_THRESHOLD # NaN -> False

# Three destabilizing groups by concordance of AM and SPURS
both_destab       = missense[am_pos & spurs_pos  & base_ok].copy()
spurs_only_destab = missense[spurs_pos & ~am_pos & base_ok].copy()
am_only_destab    = missense[am_pos & ~spurs_pos & base_ok].copy()

# Neutral control: common benign variants
neutral = missense[(missense['am_pathogenicity'] <= AM_BENIGN_MAX) &
                   (missense['gnomADe_AF'] >= GNOMAD_NEUTRAL_MIN)].copy()

for lbl, dfg in [('Both AM+SPURS', both_destab), ('SPURS only', spurs_only_destab),
                  ('AM only',      am_only_destab),  ('Neutral',      neutral)]:
    print(f'{lbl:<18}: {len(dfg):,} rows, {dfg["SYMBOL"].nunique()} genes')

# ── SINGLE-MUTATION FILTER ─────────────────────────────────────────────────────
mut_counts_per_gene = missense.groupby(['sample_id', 'SYMBOL'])['HGVSp'].nunique()
single_mut_set = set(zip(
    mut_counts_per_gene[mut_counts_per_gene == 1].index.get_level_values('sample_id'),
    mut_counts_per_gene[mut_counts_per_gene == 1].index.get_level_values('SYMBOL'),
))
print(f'\nSingle-mutation (sample, gene) pairs: {len(single_mut_set):,}')

def filter_and_build_maps(dfg, name):
    dfs = dfg[dfg.apply(lambda r: (r['sample_id'], r['SYMBOL']) in single_mut_set, axis=1)]
    sg_map  = dfs.groupby('sample_id')['SYMBOL'].apply(set).to_dict()
    vaf_lkp = (dfs[['sample_id','SYMBOL','VAF']]
               .drop_duplicates(['sample_id','SYMBOL'])
               .set_index(['sample_id','SYMBOL'])['VAF'].to_dict())
    score_lkp = (dfs[['sample_id','SYMBOL','am_pathogenicity','spurs_ddg']]
                 .drop_duplicates(['sample_id','SYMBOL'])
                 .set_index(['sample_id','SYMBOL']))
    print(f'{name} after single-mut filter: {len(dfs):,} rows, {dfs["SYMBOL"].nunique()} genes, {len(sg_map)} samples')
    return dfs, sg_map, vaf_lkp, score_lkp

both_s,  both_sg_map,  both_vaf_lkp,  both_score_lkp  = filter_and_build_maps(both_destab,       'Both AM+SPURS')
spurs_s, spurs_sg_map, spurs_vaf_lkp, spurs_score_lkp = filter_and_build_maps(spurs_only_destab, 'SPURS only')
am_s,    am_sg_map,    am_vaf_lkp,    am_score_lkp    = filter_and_build_maps(am_only_destab,    'AM only')
neutral_s, neutral_sg_map, neutral_vaf_lkp, neutral_score_lkp = filter_and_build_maps(neutral,   'Neutral')

# All-AM and all-SPURS groups for dose-response plots
all_am    = missense[missense['am_pathogenicity'].notna() & base_ok].copy()
all_spurs = missense[missense['spurs_ddg'].notna()        & base_ok].copy()
for nm, dfg in [('all_am', all_am), ('all_spurs', all_spurs)]:
    dfs = dfg[dfg.apply(lambda r: (r['sample_id'], r['SYMBOL']) in single_mut_set, axis=1)]
    sg  = dfs.groupby('sample_id')['SYMBOL'].apply(set).to_dict()
    vl  = dfs[['sample_id','SYMBOL','VAF']].drop_duplicates(['sample_id','SYMBOL']).set_index(['sample_id','SYMBOL'])['VAF'].to_dict()
    sl  = dfs[['sample_id','SYMBOL','am_pathogenicity' if nm=='all_am' else 'spurs_ddg']].drop_duplicates(['sample_id','SYMBOL']).set_index(['sample_id','SYMBOL'])
    if nm == 'all_am':
        all_am_s, all_am_sg_map, all_am_vaf_lkp, all_am_score_lkp = dfs, sg, vl, sl
    else:
        all_spurs_s, all_spurs_sg_map, all_spurs_vaf_lkp, all_spurs_score_lkp = dfs, sg, vl, sl
    print(f'{nm}: {len(dfs):,} rows, {dfs["SYMBOL"].nunique()} genes')

# keep for backwards compat in stats/save cells
destab_sample_genes  = both_sg_map  # used by preflight diagnostic
neutral_sample_genes = neutral_sg_map
vaf_lookup           = both_vaf_lkp
neutral_vaf_lookup   = neutral_vaf_lkp
score_lookup         = both_score_lkp
all_processed_uuids_placeholder = None  # defined in helpers cell


In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
import re as _re

REF_FASTA = "/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta"

def find_ri_cols(df):
    found = {}
    all_channels = CHANNEL_ORDER + ["126C","132N","132C","133N","133C","134N"]
    for ch in all_channels:
        if ch in df.columns:
            found[ch] = ch; continue
        candidates = [c for c in df.columns if c.startswith("Intensity") and c.endswith(f"_{ch}")]
        if candidates:
            found[ch] = candidates[0]
    return found

def get_channel_uuid_map(plex_id):
    pt = tmt[tmt["run_metadata_id"] == plex_id][["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates()
    pt = pt[~pt["case_submitter_id"].str.lower().isin(["ref","reference","pooled","pool","nan"])]
    pt["channel"] = pt["tmt_channel"].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=["channel"])
    merged = pt.merge(gdc[["gdc_file_id","case_submitter_id","sample_type"]],
                      on=["case_submitter_id","sample_type"], how="left")
    return merged.dropna(subset=["gdc_file_id"]).drop_duplicates("channel").set_index("channel")["gdc_file_id"].to_dict()

# Build entry_name_prefix -> gene_symbol map from UniProt FASTA
# e.g. "1433B" (from 1433B_HUMAN) -> "YWHAB" (from GN=YWHAB)
print("Building entry name -> gene symbol map from UniProt FASTA...")
_entry_to_gene = {}
with open(REF_FASTA) as _f:
    for _line in _f:
        if not _line.startswith(">"):
            continue
        _m_entry = _re.search(r"\|([A-Z0-9]+)_HUMAN\b", _line)
        _m_gene  = _re.search(r"GN=(\S+)", _line)
        if _m_entry and _m_gene:
            _entry_to_gene[_m_entry.group(1)] = _m_gene.group(1)
print(f"  {len(_entry_to_gene):,} entry->gene mappings loaded")

def gene_from_entry(entry_name):
    # Maps PSM Entry Name (e.g. 1433B_HUMAN) to gene symbol (e.g. YWHAB).
    # Returns None for -mut/-comp entries.
    if pd.isna(entry_name):
        return None
    s = str(entry_name)
    if s.endswith(("-mut", "-comp")):
        return None
    m = _re.match(r"^([A-Z0-9]+)_HUMAN", s)
    if not m:
        return None
    return _entry_to_gene.get(m.group(1), m.group(1))  # fallback to prefix if not in map

# ── BUILD GENOMICS UNIVERSE ────────────────────────────────────────────────────
all_processed_uuids = set(missense["sample_id"].unique())
print(f"UUIDs with any missense data (VEP processed): {len(all_processed_uuids):,}")
print("Helpers defined")


In [ ]:
# ── PRE-FLIGHT FUNNEL DIAGNOSTIC ──────────────────────────────────────────────
# Traces where destabilizing (sample, gene) pairs are lost at each step,
# *before* any MS data is consulted. Helps explain the final n.

print("── DESTABILIZING MUTATION FUNNEL ──────────────────────────────────────")

# Step 1: raw destabilizing pairs (before single-mut filter)
destab_pairs_all = set(zip(both_destab["sample_id"], both_destab["SYMBOL"]))
print(f"1. Destabilizing (sample, gene) pairs (VAF≥{VAF_THRESHOLD}, gnomAD<{GNOMAD_MAX}): "
      f"{len(destab_pairs_all):,}")

# Step 2: after single-mutation filter
destab_pairs_single = set(zip(both_s["sample_id"], both_s["SYMBOL"]))
print(f"2. After single-mutation-per-gene filter:      {len(destab_pairs_single):,}  "
      f"(removed {len(destab_pairs_all)-len(destab_pairs_single):,})")

# Step 3: samples present in any plex in plex_list
plex_set = set(plex_ids)
plex_uuids_all = set(tmt[tmt["run_metadata_id"].isin(plex_set)]["case_submitter_id"])
# map case_submitter_id -> gdc_file_id
gdc_lookup = gdc[["case_submitter_id","sample_type","gdc_file_id"]].dropna()
tmt_plex = (tmt[tmt["run_metadata_id"].isin(plex_set)]
            .merge(gdc_lookup, on=["case_submitter_id","sample_type"], how="left"))
plex_gdc_uuids = set(tmt_plex["gdc_file_id"].dropna())
destab_pairs_in_plex = {(s, g) for s, g in destab_pairs_single if s in plex_gdc_uuids}
print(f"3. Pairs whose sample appears in any plex:     {len(destab_pairs_in_plex):,}  "
      f"({len(plex_gdc_uuids):,} unique GDC UUIDs mapped across {len(plex_set)} plexes)")

# Step 4: samples whose UUID is in the genomics universe (VEP processed)
plex_genomics_uuids = plex_gdc_uuids & all_processed_uuids
destab_pairs_genomics = {(s, g) for s, g in destab_pairs_in_plex if s in plex_genomics_uuids}
print(f"4. Pairs whose sample has VEP-processed WXS:   {len(destab_pairs_genomics):,}  "
      f"({len(plex_genomics_uuids):,} / {len(plex_gdc_uuids):,} plex UUIDs have genomics)")

# Per-plex breakdown
print()
print("── PER-PLEX CHANNEL ACCOUNTING ────────────────────────────────────────")
print(f"{'Plex':<55} {'GDC':>5} {'VEP':>5} {'destab pairs':>14}")
for plex_id in plex_ids:
    ch_uuid = get_channel_uuid_map(plex_id)
    n_gdc = len(ch_uuid)
    cwg = {ch for ch, uuid in ch_uuid.items() if uuid in all_processed_uuids}
    n_genomics = len(cwg)
    n_pairs = sum(
        1 for ch, uuid in ch_uuid.items()
        if ch in cwg
        for gene in destab_sample_genes.get(uuid, set())
    )
    short = "_".join(plex_id.split("_")[2:5])
    print(f"  {short:<53} {n_gdc:>5} {n_genomics:>5} {n_pairs:>14}")

In [ ]:
# ── MAIN LOOP (single-pass + threaded) ───────────────────────────────────────
# Optimizations:
# 1. Read each PSM file ONCE for all 6 groups (was 6 separate reads before)
# 2. Pre-normalize RI columns once per plex (vectorized numpy)
# 3. ThreadPoolExecutor for I/O parallelism (threads share notebook globals)

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

N_JOBS = 8   # parallel plex readers; tune to available CPUs
_lock  = threading.Lock()  # for thread-safe progress printing

# Group configs: (label, sample_gene_map, vaf_lookup)
group_configs = [
    ('both_destab',  both_sg_map,      both_vaf_lkp),
    ('spurs_only',   spurs_sg_map,     spurs_vaf_lkp),
    ('am_only',      am_sg_map,        am_vaf_lkp),
    ('neutral',      neutral_sg_map,   neutral_vaf_lkp),
    ('all_spurs',    all_spurs_sg_map, all_spurs_vaf_lkp),
    ('all_am',       all_am_sg_map,    all_am_vaf_lkp),
]

def process_plex(plex_id):
    # Read PSM once, compute records for all groups. Returns dict label->list.
    results_dir = os.path.join(RESULTS_BASE, plex_id)
    psm_files = sorted(glob.glob(os.path.join(results_dir, '*_1/psm.tsv')))
    if not psm_files:
        psm_files = sorted(glob.glob(os.path.join(results_dir, 'psm.tsv')))
    if not psm_files:
        return None
    try:
        psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    except Exception:
        return None

    ri_col_map = find_ri_cols(psm)
    if not ri_col_map:
        return None

    ch_uuid = get_channel_uuid_map(plex_id)
    channels_with_genomics = {ch for ch, uuid in ch_uuid.items()
                              if uuid in all_processed_uuids}
    if len(channels_with_genomics) < 2:
        return None

    # Normalize once
    ref_mask = ~psm['Entry Name'].astype(str).str.endswith(('-mut', '-comp'))
    ref_for_norm = psm[ref_mask]
    ch_medians = {}
    for ch, col in ri_col_map.items():
        vals = ref_for_norm[col].replace(0, np.nan).dropna()
        if len(vals) > 10:
            ch_medians[ch] = vals.median()
    scale = {}
    if len(ch_medians) >= 2:
        global_med = np.median(list(ch_medians.values()))
        scale = {ch: global_med / med for ch, med in ch_medians.items()}

    ref_psm = psm[ref_mask].copy()
    ref_psm['_gene'] = ref_psm['Entry Name'].apply(gene_from_entry)
    ref_psm = ref_psm.dropna(subset=['_gene']).reset_index(drop=True)
    # Precursor intensity (MS1) for selecting best peptide
    ref_psm['_precursor'] = pd.to_numeric(ref_psm.get('Intensity', pd.Series(dtype=float)), errors='coerce').fillna(0)
    for ch, col in ri_col_map.items():
        ref_psm[f'_n_{ch}'] = (ref_psm[col] * scale.get(ch, 1.0)).where(ref_psm[col] > 0)

    plex_records = {cfg[0]: [] for cfg in group_configs}

    for label, sg_map, vaf_lkp in group_configs:
        mut_ch_gene = {}
        for ch in channels_with_genomics:
            uuid = ch_uuid.get(ch, '')
            for gene in sg_map.get(uuid, set()):
                if (uuid, gene) in single_mut_set:
                    mut_ch_gene.setdefault(gene, []).append(ch)

        wt_ch_gene = {}
        for gene, mut_chs_g in mut_ch_gene.items():
            wt_chs_g = [ch for ch in channels_with_genomics
                        if ch not in mut_chs_g and
                        gene not in sg_map.get(ch_uuid.get(ch, ''), set())]
            if len(wt_chs_g) >= 2:
                wt_ch_gene[gene] = wt_chs_g

        mut_ch_gene = {g: v for g, v in mut_ch_gene.items() if g in wt_ch_gene}
        if not mut_ch_gene:
            continue

        gene_psm = ref_psm[ref_psm['_gene'].isin(mut_ch_gene)]

        for gene, mut_chs_g in mut_ch_gene.items():
            gpsm = gene_psm[gene_psm['_gene'] == gene]
            if len(gpsm) == 0:
                continue
            wt_ncols = [f'_n_{ch}' for ch in wt_ch_gene[gene] if f'_n_{ch}' in gpsm.columns]
            if len(wt_ncols) < 2:
                continue
            wt_mat = gpsm[wt_ncols].values.astype(float)
            n_valid_wt = np.sum(np.isfinite(wt_mat), axis=1)
            with np.errstate(all='ignore'):
                wt_mean = np.where(n_valid_wt >= 2, np.nanmean(wt_mat, axis=1), np.nan)
            valid_wt = np.isfinite(wt_mean)
            peptides = gpsm['Peptide'].values if 'Peptide' in gpsm.columns else [''] * len(gpsm)

            for mut_ch in mut_chs_g:
                ncol = f'_n_{mut_ch}'
                if ncol not in gpsm.columns:
                    continue
                uuid = ch_uuid.get(mut_ch, '')
                vaf  = vaf_lkp.get((uuid, gene), np.nan) if vaf_lkp else np.nan
                mut_vals = gpsm[ncol].values.astype(float)
                mask = valid_wt & np.isfinite(mut_vals)
                if not mask.any():
                    continue
                for i in np.where(mask)[0]:
                    plex_records[label].append({
                        'plex_id':   plex_id,
                        'gene':      gene,
                        'channel':   mut_ch,
                        'sample_id': uuid,
                        'vaf':       float(vaf) if vaf == vaf else float('nan'),
                        'peptide':   str(peptides[i]),
                        'delta':     float(np.log2(mut_vals[i] / wt_mean[i])),
                        'intensity': float(mut_vals[i]),
                        'wt_n':      int(n_valid_wt[i]),
                    'precursor_intensity': float(gpsm['_precursor'].values[i]),
                    })
    return plex_records


print(f'Processing {len(plex_ids)} plexes with {N_JOBS} threads (single-pass per plex)...')
combined = {cfg[0]: [] for cfg in group_configs}
n_done = 0

with ThreadPoolExecutor(max_workers=N_JOBS) as pool:
    futures = {pool.submit(process_plex, pid): pid for pid in plex_ids}
    for fut in as_completed(futures):
        result = fut.result()
        if result:
            for label, recs in result.items():
                combined[label].extend(recs)
        n_done += 1
        if n_done % 10 == 0:
            with _lock:
                total = sum(len(v) for v in combined.values())
                print(f'  {n_done}/{len(plex_ids)} plexes done, {total:,} total records', flush=True)

df_both       = pd.DataFrame(combined['both_destab'])
df_spurs_only = pd.DataFrame(combined['spurs_only'])
df_am_only    = pd.DataFrame(combined['am_only'])
df_neutral    = pd.DataFrame(combined['neutral'])
df_all_spurs  = pd.DataFrame(combined['all_spurs'])
df_all_am     = pd.DataFrame(combined['all_am'])
df = df_both  # backwards compat

for label, df_g in [('Both AM+SPURS', df_both), ('SPURS only', df_spurs_only),
                     ('AM only', df_am_only), ('Neutral', df_neutral)]:
    print(f'{label:<18}: {len(df_g):,} PSM records')


In [ ]:
# ── GLOBAL STATISTICS ─────────────────────────────────────────────────────────
MAX_MW = 50_000  # Mann-Whitney is O(n*m) — sample for speed

d_psm = df_both["delta"].dropna()
n_psm = df_neutral["delta"].dropna()
mw_d  = d_psm.sample(min(MAX_MW, len(d_psm)), random_state=42)
mw_n  = n_psm.sample(min(MAX_MW, len(n_psm)), random_state=42)

t,    p    = stats.ttest_1samp(d_psm, popmean=0)
t_n,  p_n  = stats.ttest_1samp(n_psm, popmean=0)
u,    p_mw = stats.mannwhitneyu(mw_d, mw_n, alternative="less")

print("── PSM-LEVEL ──────────────────────────────────────────────────────────")
print(f"Both AM+SPURS: n={len(df_both):,}  mean={df_both['delta'].mean():.4f}  t={t:.2f}  p={p:.2e}")
print(f"Neutral:       n={len(df_neutral):,}  mean={df_neutral['delta'].mean():.4f}  t={t_n:.2f}  p={p_n:.2e}")
print(f"Mann-Whitney U (sampled n={MAX_MW:,}, destabilizing < neutral): p={p_mw:.2e}")

print()
print("── MUTATION COUNTS CONTRIBUTING TO ANALYSIS ──────────────────────────")
for label, df_g in [("Both AM+SPURS", df_both), ("SPURS only", df_spurs_only),
                     ("AM only", df_am_only),    ("Neutral",    df_neutral)]:
    if len(df_g) == 0: continue
    n_pairs  = df_g[["sample_id","gene"]].drop_duplicates().shape[0]
    n_genes  = df_g["gene"].nunique()
    n_plexes = df_g["plex_id"].nunique()
    print(f"{label:<18}: {n_pairs:>7,} (sample,gene) pairs | {n_genes:>5} genes | {n_plexes} plexes")


In [ ]:
# ── SPURS MATCHING DIAGNOSTIC ─────────────────────────────────────────────────
# Verifies entry name -> gene symbol mapping and SPURS coverage in plex PSMs.

import glob as _glob

print("── ENTRY NAME -> GENE SYMBOL MAPPING SPOT CHECK ───────────────────")
shown = 0
for psm_f in sorted(_glob.glob(f"{RESULTS_BASE}/*/*_1/psm.tsv") +
                    _glob.glob(f"{RESULTS_BASE}/*/psm.tsv"))[:1]:
    psm_tmp = pd.read_csv(psm_f, sep="\t", usecols=["Entry Name"], nrows=300, low_memory=False)
    for e in psm_tmp["Entry Name"].dropna():
        e = str(e)
        if e.endswith(("-mut","-comp")): continue
        mapped = gene_from_entry(e)
        print(f"  {e:<30} -> {mapped}")
        shown += 1
        if shown >= 15: break

In [ ]:
# ── COLLAPSE TO PROTEIN LEVEL ─────────────────────────────────────────────────
VAF_HOMO_MIN = 0.7

def collapse_protein(dfr, group_label):
    if len(dfr) == 0:
        return pd.DataFrame()
    grp = dfr.groupby(['plex_id', 'gene', 'channel', 'sample_id'])
    out = grp['delta'].mean().reset_index().rename(columns={'delta': 'mean_delta'})
    out['vaf'] = grp['vaf'].first().values
    out['zygosity'] = out['vaf'].apply(
        lambda v: 'homozygous' if pd.notna(v) and v > VAF_HOMO_MIN else
                  ('heterozygous' if pd.notna(v) else 'unknown'))
    if 'intensity' in dfr.columns:
        def weighted_delta(g):
            w = g['intensity'].fillna(0)
            return (g['delta'] * w).sum() / w.sum() if w.sum() > 0 else g['delta'].mean()
        out['weighted_delta'] = grp.apply(weighted_delta).values
        out['mean_intensity'] = grp['intensity'].mean().values
    out['group'] = group_label
    return out

prot_both       = collapse_protein(df_both,       'Both AM+SPURS')
prot_spurs_only = collapse_protein(df_spurs_only, 'SPURS only')
prot_am_only    = collapse_protein(df_am_only,    'AM only')
prot_neutral    = collapse_protein(df_neutral,    'Neutral')

# Combined table for faceted plots
prot_all_groups = pd.concat([prot_both, prot_spurs_only, prot_am_only, prot_neutral], ignore_index=True)

# Backwards compat aliases for downstream cells
prot_destab = prot_both

GROUP_COLORS = {'Both AM+SPURS': '#d62728', 'SPURS only': '#9467bd',
                'AM only': '#ff7f0e', 'Neutral': '#2ca02c'}
GROUP_ORDER  = ['Both AM+SPURS', 'SPURS only', 'AM only', 'Neutral']

print('Protein-level counts:')
for g in GROUP_ORDER:
    sub = prot_all_groups[prot_all_groups['group'] == g]
    homo = sub[sub['zygosity'] == 'homozygous']
    het  = sub[sub['zygosity'] == 'heterozygous']
    mn   = sub['mean_delta'].mean()
    print(f'  {g:<18}: n={len(sub):>5}  mean={mn:.4f}  homo={len(homo)}  het={len(het)}')

# Stats for backwards compat
t_p,  p_p  = stats.ttest_1samp(prot_both['mean_delta'].dropna(), popmean=0) if len(prot_both) else (np.nan, np.nan)
t_pn, p_pn = stats.ttest_1samp(prot_neutral['mean_delta'].dropna(), popmean=0) if len(prot_neutral) else (np.nan, np.nan)
u_p, p_mw_p = stats.mannwhitneyu(prot_both['mean_delta'].dropna(),
                                  prot_neutral['mean_delta'].dropna(),
                                  alternative='less') if (len(prot_both) and len(prot_neutral)) else (np.nan, np.nan)
print(f'\nMann-Whitney Both AM+SPURS < Neutral: p={p_mw_p:.2e}')


In [ ]:
# ── FACETED GROUP × ZYGOSITY PLOTS ───────────────────────────────────────────
# Panel A: violin plot — all 4 groups × zygosity side by side
# Panel B: mean delta per group bar chart stratified by zygosity

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel A: violin per group, split homo/het/neutral ─────────────────────────
ax = axes[0]
positions = []
labels    = []
all_violins = []
xtick_positions = []
xtick_labels    = []
pos = 1

for g in GROUP_ORDER:
    sub = prot_all_groups[prot_all_groups['group'] == g]
    color = GROUP_COLORS[g]
    homo_d = sub[sub['zygosity'] == 'homozygous']['mean_delta'].dropna().values
    het_d  = sub[sub['zygosity'] == 'heterozygous']['mean_delta'].dropna().values

    group_center = pos + 0.5
    xtick_positions.append(group_center)
    xtick_labels.append(g)

    for data, zy, alpha, hatch in [(homo_d, 'homo', 0.9, '//'), (het_d, 'het', 0.55, '')]:
        if len(data) >= 3:
            vp = ax.violinplot([data], positions=[pos], showmedians=True, showextrema=False, widths=0.7)
            for body in vp['bodies']:
                body.set_facecolor(color); body.set_alpha(alpha); body.set_hatch(hatch)
            vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(1.5)
            ax.annotate(f'{zy}\nn={len(data)}', (pos, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -0.5),
                        ha='center', va='top', fontsize=5.5)
        pos += 1
    pos += 0.6  # gap between groups

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(xtick_positions)
ax.set_xticklabels(xtick_labels, fontsize=8, rotation=15, ha='right')
ax.set_ylabel('protein-level mean log2 delta', fontsize=9)
ax.set_title('Group × zygosity (violin, hatched=homo)', fontsize=9)

# Add legend patches
patches = [mpatches.Patch(facecolor=GROUP_COLORS[g], label=g) for g in GROUP_ORDER]
patches += [mpatches.Patch(facecolor='grey', hatch='//', label='homozygous (VAF>0.7)'),
            mpatches.Patch(facecolor='grey', alpha=0.5, label='heterozygous')]
ax.legend(handles=patches, fontsize=6.5, loc='lower right')

# ── Panel B: mean ± SEM bar chart per group × zygosity ────────────────────────
ax2 = axes[1]
x = np.arange(len(GROUP_ORDER))
width = 0.35

for i, (zy, zy_label, offset, hatch) in enumerate(
        [('homozygous',   'Homozygous (VAF>0.7)',    -width/2, '//'),
         ('heterozygous', 'Heterozygous (0.3≤VAF≤0.7)', width/2, '')]):
    means, sems, ns = [], [], []
    for g in GROUP_ORDER:
        sub = prot_all_groups[(prot_all_groups['group'] == g) &
                               (prot_all_groups['zygosity'] == zy)]['mean_delta'].dropna()
        means.append(sub.mean() if len(sub) else np.nan)
        sems.append(sub.sem()  if len(sub) > 1 else 0)
        ns.append(len(sub))
    bars = ax2.bar(x + offset, means, width, yerr=sems, capsize=4,
                   color=[GROUP_COLORS[g] for g in GROUP_ORDER],
                   alpha=0.9 if zy == 'homozygous' else 0.55,
                   hatch=hatch, label=zy_label, error_kw={'linewidth': 1})
    for xi, (m, n) in enumerate(zip(means, ns)):
        if not np.isnan(m):
            ax2.text(xi + offset, (m or 0) + (sems[xi] or 0) + 0.01,
                     f'n={n}', ha='center', va='bottom', fontsize=5.5)

ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xticks(x)
ax2.set_xticklabels(GROUP_ORDER, fontsize=8, rotation=15, ha='right')
ax2.set_ylabel('mean log2 delta (mut / plex WT)', fontsize=9)
ax2.set_title('Mean abundance delta by group and zygosity', fontsize=9)
ax2.legend(fontsize=7)

plt.suptitle('Protein abundance reduction: AM/SPURS concordance × zygosity', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'group_zygosity_facet.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved group_zygosity_facet.pdf')


In [ ]:
# ── FACETED GROUP × ZYGOSITY — MAX INTENSITY PSM ONLY ────────────────────────
# Alternative collapse: instead of averaging all shared peptide deltas,
# use only the single PSM with the highest mutant-channel intensity per
# (plex, gene, channel, sample_id). Gives the most reliably measured peptide.

def collapse_max_intensity(dfr, group_label):
    if len(dfr) == 0:
        return pd.DataFrame()
    # For each (plex, gene, channel, sample_id), keep only the row with
    # highest precursor (MS1) intensity — best-measured shared peptide
    intensity_col = 'precursor_intensity' if 'precursor_intensity' in dfr.columns else 'intensity'
    idx = (dfr.groupby(['plex_id', 'gene', 'channel', 'sample_id'])[intensity_col]
              .idxmax())
    out = dfr.loc[idx].copy().reset_index(drop=True)
    out = out.rename(columns={'delta': 'mean_delta'})
    out['zygosity'] = out['vaf'].apply(
        lambda v: 'homozygous' if pd.notna(v) and v > VAF_HOMO_MIN else
                  ('heterozygous' if pd.notna(v) else 'unknown'))
    out['group'] = group_label
    return out[['plex_id','gene','channel','sample_id','vaf','zygosity','mean_delta','intensity','group']]

prot_both_max       = collapse_max_intensity(df_both,       'Both AM+SPURS')
prot_spurs_only_max = collapse_max_intensity(df_spurs_only, 'SPURS only')
prot_am_only_max    = collapse_max_intensity(df_am_only,    'AM only')
prot_neutral_max    = collapse_max_intensity(df_neutral,    'Neutral')
prot_max = pd.concat([prot_both_max, prot_spurs_only_max,
                      prot_am_only_max, prot_neutral_max], ignore_index=True)

print('Max-intensity PSM collapse:')
for g in GROUP_ORDER:
    sub = prot_max[prot_max['group'] == g]
    print(f'  {g:<18}: n={len(sub):>5}  mean={sub["mean_delta"].mean():.4f}  '
          f'homo={len(sub[sub["zygosity"]=="homozygous"])}  '
          f'het={len(sub[sub["zygosity"]=="heterozygous"])}')

# ── Same violin + bar layout as cell 10 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
pos = 1
xtick_positions, xtick_labels = [], []
for g in GROUP_ORDER:
    sub = prot_max[prot_max['group'] == g]
    color = GROUP_COLORS[g]
    xtick_positions.append(pos + 0.5); xtick_labels.append(g)
    for data, zy, alpha, hatch in [
        (sub[sub['zygosity']=='homozygous']['mean_delta'].dropna().values,  'homo', 0.9, '//'),
        (sub[sub['zygosity']=='heterozygous']['mean_delta'].dropna().values, 'het', 0.55, ''),
    ]:
        if len(data) >= 3:
            vp = ax.violinplot([data], positions=[pos], showmedians=True,
                               showextrema=False, widths=0.7)
            for body in vp['bodies']:
                body.set_facecolor(color); body.set_alpha(alpha); body.set_hatch(hatch)
            vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(1.5)
        pos += 1
    pos += 0.6

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(xtick_positions)
ax.set_xticklabels(xtick_labels, fontsize=8, rotation=15, ha='right')
ax.set_ylabel('log2 delta (max-intensity PSM)', fontsize=9)
ax.set_title('Group × zygosity — max intensity PSM only\n(hatched=homo)', fontsize=9)
patches = [mpatches.Patch(facecolor=GROUP_COLORS[g], label=g) for g in GROUP_ORDER]
patches += [mpatches.Patch(facecolor='grey', hatch='//', label='homozygous'),
            mpatches.Patch(facecolor='grey', alpha=0.5, label='heterozygous')]
ax.legend(handles=patches, fontsize=6.5, loc='lower right')

ax2 = axes[1]
x = np.arange(len(GROUP_ORDER)); width = 0.35
for i, (zy, zy_label, offset, hatch) in enumerate([
        ('homozygous',   'Homozygous (VAF>0.7)',       -width/2, '//'),
        ('heterozygous', 'Heterozygous (0.3≤VAF≤0.7)',  width/2, ''),
]):
    means, sems, ns = [], [], []
    for g in GROUP_ORDER:
        sub = prot_max[(prot_max['group']==g) & (prot_max['zygosity']==zy)]['mean_delta'].dropna()
        means.append(sub.mean() if len(sub) else np.nan)
        sems.append(sub.sem()   if len(sub) > 1 else 0)
        ns.append(len(sub))
    ax2.bar(x + offset, means, width, yerr=sems, capsize=4,
            color=[GROUP_COLORS[g] for g in GROUP_ORDER],
            alpha=0.9 if zy == 'homozygous' else 0.55,
            hatch=hatch, label=zy_label, error_kw={'linewidth': 1})
    for xi, (m, n) in enumerate(zip(means, ns)):
        if not np.isnan(m):
            ax2.text(xi + offset, (m or 0) + (sems[xi] or 0) + 0.01,
                     f'n={n}', ha='center', va='bottom', fontsize=5.5)

ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xticks(x)
ax2.set_xticklabels(GROUP_ORDER, fontsize=8, rotation=15, ha='right')
ax2.set_ylabel('mean log2 delta (max-intensity PSM)', fontsize=9)
ax2.set_title('Mean delta by group × zygosity\n(max-intensity PSM)', fontsize=9)
ax2.legend(fontsize=7)

plt.suptitle('Protein abundance: concordance × zygosity — max intensity PSM only', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'group_zygosity_maxpsm.pdf'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── SCORE vs PROTEIN DELTA PLOTS ─────────────────────────────────────────────
# Both plots use their respective 'all_*' groups: every single-mutation
# observation with the relevant score and VAF>=threshold, no gnomAD/AM/SPURS
# cross-filter. Gives a true dose-response across the full score range.

# Collapse all_spurs and all_am to protein level
prot_all_spurs = collapse_protein(df_all_spurs, 'all_spurs')
prot_all_spurs['spurs_ddg'] = prot_all_spurs.apply(
    lambda r: all_spurs_score_lookup.loc[(r['sample_id'], r['gene']), 'spurs_ddg']
    if (r['sample_id'], r['gene']) in all_spurs_score_lookup.index else np.nan, axis=1)
print(f'All-SPURS protein-level observations: {len(prot_all_spurs):,}')
print(f'SPURS ddG range: {prot_all_spurs["spurs_ddg"].min():.3f} – '
      f'{prot_all_spurs["spurs_ddg"].max():.3f}')

prot_all_am = collapse_protein(df_all_am, 'all_am')
prot_all_am['am_pathogenicity'] = prot_all_am.apply(
    lambda r: all_am_score_lookup.loc[(r['sample_id'], r['gene']), 'am_pathogenicity']
    if (r['sample_id'], r['gene']) in all_am_score_lookup.index else np.nan, axis=1)
print(f'All-AM protein-level observations: {len(prot_all_am):,}')
print(f'AM score range: {prot_all_am["am_pathogenicity"].min():.3f} – '
      f'{prot_all_am["am_pathogenicity"].max():.3f}')

def bin_score_plot(ax, data, score_col, score_label, color, n_bins=8):
    """Bin data by score_col into n_bins quantile bins, plot mean±SEM of mean_delta."""
    d = data[[score_col, 'mean_delta']].dropna()
    if len(d) < n_bins * 3:
        ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center',
                transform=ax.transAxes)
        return
    d = d.copy()
    d['bin'] = pd.qcut(d[score_col], q=n_bins, duplicates='drop')
    grouped = d.groupby('bin', observed=True)['mean_delta']
    means  = grouped.mean()
    sems   = grouped.sem()
    ns     = grouped.count()
    bin_mids = [iv.mid for iv in means.index]

    ax.errorbar(bin_mids, means.values, yerr=sems.values,
                fmt='o-', color=color, capsize=4, lw=1.5, ms=5)
    for x, y, n in zip(bin_mids, means.values, ns.values):
        ax.annotate(f'n={n}', (x, y), textcoords='offset points',
                    xytext=(0, 7), ha='center', fontsize=6)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel(score_label, fontsize=9)
    ax.set_ylabel('mean log2 delta (mut / plex WT)', fontsize=8)
    r, p_r = stats.pearsonr(d[score_col], d['mean_delta'])
    ax.set_title(f'{score_label} vs abundance delta\nr={r:.3f}, p={p_r:.2e}\n'
                 f'n={len(d):,} protein observations', fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bin_score_plot(axes[0], prot_all_spurs, 'spurs_ddg',
               'SPURS ddG (full range, all scored mutations)',
               '#d62728', n_bins=8)

bin_score_plot(axes[1], prot_all_am, 'am_pathogenicity',
               'AlphaMissense pathogenicity (full range, all scored mutations)',
               '#e07b39', n_bins=8)

plt.suptitle('Predicted destabilization score vs observed protein abundance change\n'
             '(all mutations with respective score + VAF≥0.3, single-mutation samples)',
             fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'score_vs_delta.pdf'), dpi=150)
plt.show()


In [ ]:
# ── PLOT: protein-level delta distributions — all 4 groups ───────────────────
all_deltas_combined = prot_all_groups['mean_delta'].dropna()
q01 = all_deltas_combined.quantile(0.01)
q99 = all_deltas_combined.quantile(0.99)
bins = np.linspace(q01, q99, 80)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)

for ax, g in zip(axes, GROUP_ORDER):
    data = prot_all_groups[prot_all_groups['group'] == g]['mean_delta'].dropna()
    color = GROUP_COLORS[g]
    t_val, p_val = stats.ttest_1samp(data, popmean=0) if len(data) >= 3 else (np.nan, np.nan)
    ax.hist(data, bins=bins, color=color, alpha=0.75, edgecolor='none')
    mean_d = data.mean()
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.axvline(mean_d, color='k', lw=1.5, ls='-',
               label=f'mean={mean_d:.3f}\nt={t_val:.2f}, p={p_val:.1e}\nn={len(data):,}')
    ax.set_xlabel('mean log2(mut / plex WT)', fontsize=8)
    ax.set_ylabel('protein observations', fontsize=8)
    ax.set_title(g, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle(f'Protein abundance delta: concordance groups (protein-level)\n'
             f'Mann-Whitney Both AM+SPURS < Neutral: p={p_mw_p:.2e}', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'all_groups_dist.pdf'), dpi=150)
plt.show()


In [ ]:
# ── PER-PLEX HISTOGRAMS ───────────────────────────────────────────────────────
# One subplot per TMT multiplexed set, showing destabilizing (red) and
# neutral (green) protein-level delta distributions side by side.
from matplotlib.backends.backend_pdf import PdfPages

MIN_N_PER_PLEX = 3   # skip plexes with fewer protein observations than this

all_plexes = sorted(set(prot_destab["plex_id"].tolist() + prot_neutral["plex_id"].tolist()))
plexes_to_plot = [p for p in all_plexes
                  if (len(prot_destab[prot_destab["plex_id"] == p]) >= MIN_N_PER_PLEX or
                      len(prot_neutral[prot_neutral["plex_id"] == p]) >= MIN_N_PER_PLEX)]
print(f"Plexes with data (≥{MIN_N_PER_PLEX} observations): {len(plexes_to_plot)}")

NCOLS = 5
nrows = -(-len(plexes_to_plot) // NCOLS)   # ceiling division

fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.5, nrows * 2.8))
axes_flat = axes.flatten() if nrows > 1 else list(axes) if NCOLS > 1 else [axes]

# Compute shared x-axis limits from 1st/99th percentiles across all data
all_deltas = pd.concat([prot_destab["mean_delta"], prot_neutral["mean_delta"]]).dropna()
xlim = max(abs(all_deltas.quantile(0.01)), abs(all_deltas.quantile(0.99)))
xlim = min(xlim, 5.0)
bins = np.linspace(-xlim, xlim, 25)

for ax, plex_id in zip(axes_flat, plexes_to_plot):
    d = prot_destab[prot_destab["plex_id"] == plex_id]["mean_delta"].dropna()
    n = prot_neutral[prot_neutral["plex_id"] == plex_id]["mean_delta"].dropna()

    if len(d):
        ax.hist(d, bins=bins, color="#d62728", alpha=0.65, label=f"destab n={len(d)}")
        ax.axvline(d.mean(), color="#d62728", lw=1.2, ls="-")
    if len(n):
        ax.hist(n, bins=bins, color="#2ca02c", alpha=0.65, label=f"neutral n={len(n)}")
        ax.axvline(n.mean(), color="#2ca02c", lw=1.2, ls="-")

    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_xlim(-xlim, xlim)

    # Short label: tissue + date from plex ID (e.g. CCRCC_JHU_20171007)
    parts = plex_id.split("_")
    short = "_".join(parts[2:5]) if len(parts) >= 5 else plex_id[:22]
    ax.set_title(short, fontsize=5.5, pad=2)
    ax.tick_params(labelsize=5)
    ax.legend(fontsize=4.5, loc="upper right", framealpha=0.5)
    ax.set_xlabel("mean log2 delta", fontsize=5)

for ax in axes_flat[len(plexes_to_plot):]:
    ax.set_visible(False)

plt.suptitle(f"Per-plex protein-level delta: destabilizing vs neutral\n"
             f"(only proteins with single mutation in sample, ≥{MIN_N_PER_PLEX} obs per plex)",
             fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_plex_histograms.pdf"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved per-plex histogram PDF: {OUT_DIR}/per_plex_histograms.pdf")

In [ ]:
# ── INTENSITY-BINNED DELTA PLOT (destabilizing) ───────────────────────────────
# Each protein observation is weighted by its mean normalized mut-channel
# intensity across contributing PSMs. Observations are then binned into 5
# quantiles by that mean intensity, and we plot the intensity-weighted mean
# delta ± SEM per bin.
# Tests whether the abundance effect is consistent across intensity strata
# (low intensity = noisy, high intensity = precise measurement).

d = prot_destab[['mean_intensity', 'weighted_delta', 'mean_delta']].dropna()
d = d[d['mean_intensity'] > 0].copy()
d['log_intensity'] = np.log10(d['mean_intensity'])
d['bin'] = pd.qcut(d['log_intensity'], q=5, duplicates='drop')

grouped = d.groupby('bin', observed=True)
means_w  = grouped['weighted_delta'].mean()
sems_w   = grouped['weighted_delta'].sem()
means_uw = grouped['mean_delta'].mean()
sems_uw  = grouped['mean_delta'].sem()
ns       = grouped['weighted_delta'].count()
bin_mids = [iv.mid for iv in means_w.index]
bin_labels = [f'{10**iv.left:.0f}–{10**iv.right:.0f}' for iv in means_w.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, means, sems, ylabel, title_suffix in [
    (axes[0], means_w,  sems_w,  'intensity-weighted mean log2 delta', 'Intensity-weighted'),
    (axes[1], means_uw, sems_uw, 'unweighted mean log2 delta',         'Unweighted'),
]:
    ax.errorbar(range(len(means)), means.values, yerr=sems.values,
                fmt='o-', color='#d62728', capsize=4, lw=1.5, ms=6)
    for x, y, n in zip(range(len(means)), means.values, ns.values):
        ax.annotate(f'n={n}', (x, y), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(range(len(bin_labels)))
    ax.set_xticklabels(bin_labels, rotation=30, ha='right', fontsize=7)
    ax.set_xlabel('mean normalized mut-channel intensity (quantile bins)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(f'{title_suffix} delta by intensity quintile\n(destabilizing mutations)', fontsize=9)

plt.suptitle('Does the abundance effect hold across intensity strata?\n'
             '(left: delta weighted by peptide intensity | right: unweighted for comparison)',
             fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'intensity_binned_delta.pdf'), dpi=150)
plt.show()


In [ ]:
# ── PER-GENE STATS (destabilizing, protein-level) ─────────────────────────────
from statsmodels.stats.multitest import multipletests

gene_stats = []
for gene, grp in prot_destab.groupby("gene"):
    deltas = grp["mean_delta"].dropna().values
    if len(deltas) < 3:
        continue
    t_g, p_g = stats.ttest_1samp(deltas, popmean=0)
    gene_stats.append({"gene": gene, "n_proteins": len(deltas), "n_plexes": grp["plex_id"].nunique(),
                       "mean_delta": np.mean(deltas), "t_stat": t_g, "p_value": p_g})

gene_stats = pd.DataFrame(gene_stats).sort_values("p_value")
if len(gene_stats):
    _, gene_stats["fdr"], _, _ = multipletests(gene_stats["p_value"], method="fdr_bh")

sig = (gene_stats["fdr"] < 0.05) & (gene_stats["mean_delta"] < 0)
print(f"Genes tested: {len(gene_stats)} | Significant (FDR<0.05, delta<0): {sig.sum()}")
gene_stats.head(20)

In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df_both.to_csv(os.path.join(OUT_DIR, 'both_destab_per_psm.tsv'), sep='\t', index=False)
df_spurs_only.to_csv(os.path.join(OUT_DIR, 'spurs_only_per_psm.tsv'), sep='\t', index=False)
df_am_only.to_csv(os.path.join(OUT_DIR, 'am_only_per_psm.tsv'), sep='\t', index=False)
df_neutral.to_csv(os.path.join(OUT_DIR, 'neutral_per_psm.tsv'), sep='\t', index=False)
prot_all_groups.to_csv(os.path.join(OUT_DIR, 'all_groups_per_protein.tsv'), sep='\t', index=False)
gene_stats.to_csv(os.path.join(OUT_DIR, 'gene_destab_stats.tsv'), sep='\t', index=False)
print(f'Saved to {OUT_DIR}')
